<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/02_convert_iis3dwb_domain_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import scipy.io as sio
from scipy.signal import butter, sosfilt, sosfiltfilt, resample_poly, welch
from dataclasses import dataclass, field
from typing import Optional

# ------------------------------------------------------------------------------
# 0. 상수 (명세서 hardware 섹션과 1:1 일치 — 펌웨어 헤더와 공유될 값)
# ------------------------------------------------------------------------------
SRC_FS          = 64000            # Paderborn vibration 원본 샘플레이트 [Hz]
TARGET_FS_NUM   = 80000            # IIS3DWB ODR 분자 (80000/3 = 26666.6667Hz)
TARGET_FS_DEN   = 3
TARGET_FS       = TARGET_FS_NUM / TARGET_FS_DEN   # 26666.6667 Hz

RESAMPLE_UP     = 5                # ★ 64000 * (5/12) = 26666.6667 정확히 일치
RESAMPLE_DOWN   = 12              #    64000/26666.6667 = 2.4 = 12/5 → up=5, down=12

SENSOR_BW_HZ    = 6000            # IIS3DWB 대역폭 (~6kHz). LPF cutoff.
LPF_ORDER       = 6              # 6차 Butterworth

FULL_SCALE_G    = 16.0           # ±16g
SENSITIVITY     = 0.000488       # 0.488 mg/LSB (16g / 2^15)
INT16_MIN       = -32768
INT16_MAX       = 32767

G_PER_MS2       = 1.0 / 9.80665  # m/s² → g 변환계수


# ------------------------------------------------------------------------------
# 1. 도메인 랜덤화 설정 (train 전용. val/test는 randomize=False로 호출)
# ------------------------------------------------------------------------------
@dataclass
class DomainConfig:
    # BW / AAF
    lpf_cutoff_hz: float = SENSOR_BW_HZ
    lpf_causal: bool = True          # True=sosfilt(인과, 펌웨어와 유사) / False=sosfiltfilt(무위상)

    # train-only randomization 범위
    randomize: bool = False
    lpf_cutoff_range: tuple = (5500.0, 6300.0)
    gain_range: tuple = (0.8, 1.2)
    polarity_prob: float = 0.5
    noise_rms_range_g: tuple = (0.003, 0.008)
    offset_range_g: tuple = (-1.0, 1.0)

    # 고정 파라미터
    apply_offset_gravity: bool = True
    seed: Optional[int] = None


# ------------------------------------------------------------------------------
# 2. Paderborn MAT 파서 — 중첩 struct 방어적 접근
#    구조: mat[<파일명키>].Y[i].Name / .Data  (Name에 'vibration_1' 포함)
# ------------------------------------------------------------------------------
def load_paderborn_vibration(mat_path: str, channel_name: str = "vibration_1"):
    """
    Paderborn MAT에서 지정 채널의 연속 1D 신호를 float64로 반환.
    반환: (signal_1d, meta_dict) 또는 (None, reason)
    """
    try:
        mat = sio.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    except Exception as e:
        return None, f"loadmat 실패: {e}"

    # 최상위 struct 키 찾기 (__ 로 시작하는 메타키 제외)
    top_keys = [k for k in mat.keys() if not k.startswith("__")]
    if not top_keys:
        return None, "최상위 struct 키 없음"
    root = mat[top_keys[0]]

    # Y 배열 접근 (채널 목록)
    if not hasattr(root, "Y"):
        return None, f"'Y' 필드 없음 (필드: {dir(root)})"
    Y = root.Y

    # Y가 단일 객체면 리스트화
    Y_list = Y if isinstance(Y, np.ndarray) else np.array([Y])

    target = None
    for ch in np.atleast_1d(Y_list):
        name = str(getattr(ch, "Name", "")).strip().lower()
        if channel_name.lower() in name:
            target = ch
            break
    if target is None:
        avail = [str(getattr(c, "Name", "?")) for c in np.atleast_1d(Y_list)]
        return None, f"'{channel_name}' 채널 없음. 가용: {avail}"

    sig = np.asarray(target.Data, dtype=np.float64).ravel()
    if sig.size == 0:
        return None, "채널 데이터 비어있음"

    meta = {
        "source_file": mat_path.split("/")[-1],
        "n_samples": int(sig.size),
        "duration_s": float(sig.size / SRC_FS),
        "channel": channel_name,
    }
    return sig, meta


# ------------------------------------------------------------------------------
# 3. 단위 확정 (m/s² vs g) — ★ 명세서 P2. 임의 클리핑 금지
#    휴리스틱: 정상 베어링 진동 g-RMS는 통상 0.1~2g 수준.
#    RMS가 수~수십이면 m/s² 가능성 높음. 단, 최종은 사용자/데이터시트 확인 권장.
# ------------------------------------------------------------------------------
def confirm_unit_to_g(signal: np.ndarray, declared_unit: Optional[str] = None):
    """
    declared_unit이 명시되면 그대로 신뢰. 없으면 휴리스틱 추정 + 경고.
    반환: (signal_in_g, unit_report_dict)
    """
    rms = float(np.sqrt(np.mean(signal ** 2)))
    peak = float(np.max(np.abs(signal)))
    report = {"raw_rms": rms, "raw_peak": peak, "declared_unit": declared_unit}

    if declared_unit is not None:
        u = declared_unit.lower()
        if u in ("m/s2", "m/s^2", "ms2", "mps2"):
            report["applied"] = "m/s² → g (/9.80665)"
            return signal * G_PER_MS2, report
        elif u == "g":
            report["applied"] = "이미 g. 변환 없음"
            return signal, report
        else:
            report["applied"] = f"미지정 단위 '{declared_unit}'. 변환 없음(경고)"
            return signal, report

    # 휴리스틱 (declared_unit 없을 때만)
    if rms > 5.0:  # g로 보기엔 과대 → m/s² 의심
        report["heuristic"] = f"RMS={rms:.2f} 과대 → m/s² 추정, /9.80665 적용"
        report["applied"] = "heuristic m/s² → g"
        report["WARNING"] = "★ 단위 자동추정. 데이터시트로 반드시 최종확인 요망"
        return signal * G_PER_MS2, report
    else:
        report["heuristic"] = f"RMS={rms:.2f} → g 추정, 변환 없음"
        report["applied"] = "heuristic g (변환없음)"
        report["WARNING"] = "★ 단위 자동추정. 데이터시트로 반드시 최종확인 요망"
        return signal, report


# ------------------------------------------------------------------------------
# 4. 6kHz LPF (센서 BW 모사 + 리샘플 AAF 겸용) — ★ 전체 연속신호에 1회
# ------------------------------------------------------------------------------
def apply_sensor_lpf(signal: np.ndarray, cutoff_hz: float, causal: bool = True):
    nyq = SRC_FS / 2.0
    wn = cutoff_hz / nyq
    if not (0 < wn < 1):
        raise ValueError(f"cutoff {cutoff_hz}Hz가 원본 나이퀴스트({nyq}Hz) 범위 밖")
    sos = butter(LPF_ORDER, wn, btype="low", output="sos")
    if causal:
        return sosfilt(sos, signal)      # 인과 필터 (펌웨어 실시간과 유사한 위상특성)
    else:
        return sosfiltfilt(sos, signal)  # 무위상 (분석/비교용)


# ------------------------------------------------------------------------------
# 5. Polyphase 리샘플링 64000 → 80000/3 Hz — ★ 단순 decimation 금지
# ------------------------------------------------------------------------------
def resample_to_iis3dwb(signal: np.ndarray):
    # resample_poly 내부에 Kaiser-windowed FIR AAF 포함 (2차 안전장치)
    out = resample_poly(signal, up=RESAMPLE_UP, down=RESAMPLE_DOWN)
    return out


# ------------------------------------------------------------------------------
# 6. 센서 비이상성 주입: gain / offset·gravity / band-limited noise
# ------------------------------------------------------------------------------
def _band_limited_noise(n: int, target_rms_g: float, fs: float, rng: np.random.Generator,
                        band_hz: float = SENSOR_BW_HZ):
    """대역제한 가우시안 노이즈. 생성 후 실측 RMS를 target에 정확히 정합."""
    white = rng.standard_normal(n)
    # 센서 대역(~6kHz)로 노이즈도 대역제한
    nyq = fs / 2.0
    wn = min(band_hz / nyq, 0.99)
    sos = butter(4, wn, btype="low", output="sos")
    colored = sosfilt(sos, white)
    cur_rms = np.sqrt(np.mean(colored ** 2)) + 1e-12
    return colored * (target_rms_g / cur_rms)   # ★ RMS 정확 정합


def inject_sensor_effects(signal_g: np.ndarray, cfg: DomainConfig, fs: float,
                          rng: np.random.Generator):
    meta = {}
    x = signal_g.copy()

    # (a) gain / sensitivity variation
    if cfg.randomize:
        gain = rng.uniform(*cfg.gain_range)
    else:
        gain = 1.0
    x = x * gain
    meta["gain_factor"] = float(gain)

    # (b) polarity inversion (설치 방향 반전 모사)
    if cfg.randomize and rng.random() < cfg.polarity_prob:
        x = -x
        meta["polarity"] = -1
    else:
        meta["polarity"] = 1

    # (c) offset / gravity 성분
    if cfg.apply_offset_gravity:
        if cfg.randomize:
            offset = rng.uniform(*cfg.offset_range_g)
        else:
            offset = 0.0
        x = x + offset
        meta["offset_g"] = float(offset)
    else:
        meta["offset_g"] = 0.0

    # (d) band-limited noise
    if cfg.randomize:
        target_rms = rng.uniform(*cfg.noise_rms_range_g)
    else:
        target_rms = cfg.noise_rms_range_g[0]   # val/test는 하한 고정(재현성)
    noise = _band_limited_noise(len(x), target_rms, fs, rng)
    x = x + noise
    meta["noise_rms_g"] = float(np.sqrt(np.mean(noise ** 2)))

    return x, meta


# ------------------------------------------------------------------------------
# 7. ±16g 클리핑 + INT16 양자화 (0.488mg/LSB)
# ------------------------------------------------------------------------------
def clip_and_quantize_int16(signal_g: np.ndarray):
    # ±16g 클리핑
    clip_mask = np.abs(signal_g) > FULL_SCALE_G
    n_clip = int(np.sum(clip_mask))
    clipped = np.clip(signal_g, -FULL_SCALE_G, FULL_SCALE_G)

    # g → INT16 count
    counts = np.round(clipped / SENSITIVITY).astype(np.int64)
    counts = np.clip(counts, INT16_MIN, INT16_MAX).astype(np.int16)

    q_report = {
        "clip_count": n_clip,
        "clip_ratio": float(n_clip / len(signal_g)),
        "int16_min": int(counts.min()),
        "int16_max": int(counts.max()),
    }
    return counts, q_report


# ------------------------------------------------------------------------------
# 8. 전체 파이프라인 오케스트레이션 (연속신호 → 연속 INT16)
# ------------------------------------------------------------------------------
@dataclass
class ConversionResult:
    counts_int16: np.ndarray            # 최종 연속 INT16 신호 (윈도우 이전)
    target_fs: float
    unit_report: dict = field(default_factory=dict)
    dc_report: dict = field(default_factory=dict)
    effect_meta: dict = field(default_factory=dict)
    quant_report: dict = field(default_factory=dict)
    verify: dict = field(default_factory=dict)


def convert_file_to_iis3dwb(mat_path: str,
                            cfg: DomainConfig,
                            declared_unit: Optional[str] = None,
                            channel: str = "vibration_1") -> Optional[ConversionResult]:
    rng = np.random.default_rng(cfg.seed)

    # [1] 연속신호 로드
    sig, meta = load_paderborn_vibration(mat_path, channel)
    if sig is None:
        print(f"  [SKIP] {mat_path}: {meta}")
        return None

    # [2] 단위 g 통일 (P2)
    sig_g, unit_report = confirm_unit_to_g(sig, declared_unit)

    # [3] DC/offset 확인 (제거는 안 함 — 펌웨어 window mean subtraction과 일치시키려 후단 유지)
    dc_report = {
        "mean_g": float(np.mean(sig_g)),
        "std_g": float(np.std(sig_g)),
    }

    # [4] 랜덤화 시 LPF cutoff 흔들기
    cutoff = (rng.uniform(*cfg.lpf_cutoff_range) if cfg.randomize else cfg.lpf_cutoff_hz)

    # [4] 6kHz LPF (AAF 겸용) — 전체 연속신호에 1회
    sig_lpf = apply_sensor_lpf(sig_g, cutoff, causal=cfg.lpf_causal)

    # [5] polyphase resample 64k → 80000/3
    sig_rs = resample_to_iis3dwb(sig_lpf)

    # [6] 센서 효과 주입
    sig_fx, effect_meta = inject_sensor_effects(sig_rs, cfg, TARGET_FS, rng)
    effect_meta["filter_cutoff"] = float(cutoff)

    # [7] ±16g clip + INT16 양자화
    counts, quant_report = clip_and_quantize_int16(sig_fx)

    # [검증] 변환 전후 통계 자동비교 (명세서 reproducibility 요구)
    verify = _verify_conversion(sig_g, counts)

    return ConversionResult(
        counts_int16=counts,
        target_fs=TARGET_FS,
        unit_report=unit_report,
        dc_report=dc_report,
        effect_meta=effect_meta,
        quant_report=quant_report,
        verify=verify,
    )


# ------------------------------------------------------------------------------
# 9. 변환 검증: 리샘플 비율, RMS/Peak 보존, 클리핑 과다 여부
# ------------------------------------------------------------------------------
def _verify_conversion(src_g: np.ndarray, counts_int16: np.ndarray):
    out_g = counts_int16.astype(np.float64) * SENSITIVITY
    exp_len = int(round(len(src_g) * RESAMPLE_UP / RESAMPLE_DOWN))
    v = {
        "src_len": len(src_g),
        "out_len": len(counts_int16),
        "expected_out_len": exp_len,
        "len_ok": abs(len(counts_int16) - exp_len) <= 2,
        "src_rms_g": float(np.sqrt(np.mean(src_g ** 2))),
        "out_rms_g": float(np.sqrt(np.mean(out_g ** 2))),
        "src_peak_g": float(np.max(np.abs(src_g))),
        "out_peak_g": float(np.max(np.abs(out_g))),
    }
    # 경고 플래그
    v["warn_high_clip"] = None
    return v


# ------------------------------------------------------------------------------
# 10. 사용 예시
# ------------------------------------------------------------------------------
if __name__ == "__main__":
    MAT = "/content/drive/My Drive/Colab Notebooks/N09_M07_F10_K001_1.mat"

    # ─ train 샘플 (도메인 랜덤화 ON) ─
    cfg_train = DomainConfig(randomize=True, seed=42)
    # ─ val/test 샘플 (랜덤화 OFF, 재현성) ─
    cfg_eval = DomainConfig(randomize=False, seed=42)

    # ★ declared_unit: 데이터시트/논문으로 확인되면 "m/s2" 또는 "g" 명시.
    #    미확인 시 None → 휴리스틱 + 경고 (P2 미해소 상태로 간주, 최종학습 전 반드시 확정)
    res = convert_file_to_iis3dwb(MAT, cfg_eval, declared_unit=None)

    if res:
        print("=" * 60)
        print("도메인 변환 결과 요약")
        print("=" * 60)
        print(f" 단위 : {res.unit_report.get('applied')}")
        if "WARNING" in res.unit_report:
            print(f"   {res.unit_report['WARNING']}")
        print(f" DC   : mean={res.dc_report['mean_g']:.4f}g std={res.dc_report['std_g']:.4f}g")
        print(f" 효과 : gain={res.effect_meta['gain_factor']:.3f} "
              f"pol={res.effect_meta['polarity']} "
              f"offset={res.effect_meta['offset_g']:.3f}g "
              f"noise_rms={res.effect_meta['noise_rms_g']:.4f}g "
              f"cutoff={res.effect_meta['filter_cutoff']:.0f}Hz")
        print(f" 양자화: clip={res.quant_report['clip_count']} "
              f"({res.quant_report['clip_ratio']*100:.3f}%) "
              f"range=[{res.quant_report['int16_min']},{res.quant_report['int16_max']}]")
        print(f" 검증 : src_len={res.verify['src_len']} → out_len={res.verify['out_len']} "
              f"(기대 {res.verify['expected_out_len']}, ok={res.verify['len_ok']})")
        print(f"        RMS {res.verify['src_rms_g']:.4f}g → {res.verify['out_rms_g']:.4f}g")
        print(f"        Peak {res.verify['src_peak_g']:.4f}g → {res.verify['out_peak_g']:.4f}g")
        print(f" fs   : {res.target_fs:.4f} Hz")

        if res.quant_report["clip_ratio"] > 0.001:
            print("\n ⚠ 클리핑 비율 0.1% 초과 — offset/gain 범위 재검토 필요")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
도메인 변환 결과 요약
 단위 : heuristic g (변환없음)
   ★ 단위 자동추정. 데이터시트로 반드시 최종확인 요망
 DC   : mean=-0.0155g std=0.3673g
 효과 : gain=1.000 pol=1 offset=0.000g noise_rms=0.0030g cutoff=6000Hz
 양자화: clip=0 (0.000%) range=[-3635,3032]
 검증 : src_len=256823 → out_len=107010 (기대 107010, ok=True)
        RMS 0.3676g → 0.1689g
        Peak 4.8798g → 1.7739g
 fs   : 26666.6667 Hz
